In [ ]:
from datetime import date

from flights.evaluation.evaluation import main

outputs = main("algorithm", "sfdps", test_dates=date(2026, 3, 1), adsb_src="adsblol")

In [ ]:
import polars as pl

In [ ]:
df = outputs[0][0]
COLUMNS = ["icao", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident", "takeoff_time_df1", "landing_time_df1", "takeoff_airport_ident_df1", "landing_airport_ident_df1", "match_status"]
df = df.select(COLUMNS)

In [ ]:
df.filter(pl.col("match_status") != "both")

In [ ]:
df.filter(pl.col("match_status") != "both").get_column("icao").unique().sort().to_list()

## Manual review of ADSB.lol algorithm mismatches

The complete ADSB.lol algorithm output (`df0`), SFDPS flights (`df1`), raw ADSB.lol trajectories, and the independent ADSB-X algorithm trajectories were reviewed for 2026-03-01. The evaluation contains **95 mismatched ICAOs** and **182 mismatched rows**.

Evaluation statistics: **6,611 true positives, 65 false positives, and 117 false negatives**.

### Summary

- **Gold/df1 problem:** 18 ICAOs
- **Genuine ADSB.lol algorithm/df0 failure:** 75 ICAOs
- **Both datasets have problems:** 2 ICAOs (`a5a744`, `a654ec`)

### Mismatch error percentages

Percentages use the 95 mismatched ICAOs as the denominator.

| Test set Error (%) | Gold-Dataset Error (%) | Error in both (%) |
|---:|---:|---:|
| 78.95% | 18.95% | 2.11% |

A source-coverage failure is counted as a `df0` failure here: even if ADSB.lol omitted part of the trajectory, the resulting algorithm flight is still missing, merged, or assigned incorrect endpoints.

### Gold/df1 is incorrect

| ICAO | Reason |
|---|---|
| `a1492e` | SFDPS omits a real local KCRG flight around 18:51-20:01. |
| `a1a4a7` | SFDPS combines the real KBHM-KNEW and KNEW-KBHM legs. |
| `a1a7ad` | SFDPS combines KAVL-KTOC-KAVL activity; the intermediate event is partly inside an ADS-B gap. |
| `a1a7dd` | SFDPS combines KPWA-KHSD-KPWA activity; the intermediate event is partly inside an ADS-B gap. |
| `a1c2e3` | The aircraft stays on the ground at KAPF until about 20:33; df1 claims an 18:19 departure. |
| `a1e6b9` | The aircraft stays at KHOU until the real 13:37 takeoff; df1 says 12:34. |
| `a3a1dc` | The KRBD-KMKY leg departs around 21:10; df1 incorrectly reuses 18:11. |
| `a3e348` | The FA54-KHOU leg departs around 17:57; df1 incorrectly reuses 15:07. |
| `a63c0b` | The KENW-KBNA leg departs around 16:25; df1 incorrectly reuses 12:48. |
| `a6ea99` | The aircraft remains at KGYY until approximately 16:36; df1 says 15:30. |
| `a84a06` | SFDPS combines KCHS-K38J-KCHS activity; the intermediate event is partly inside an ADS-B gap. |
| `a875ab` | The KBKL-KLNS leg departs around 10:38; df1 incorrectly reuses 06:12. |
| `a9b2c8` | SFDPS omits a real local KFMY flight. |
| `aafb20` | SFDPS omits a real local KDTO flight. |
| `abfc78` | SFDPS combines the real KSAV-KAQX and KAQX-KSAV legs. |
| `ac97c2` | SFDPS combines KJYO-WV67-KJYO activity; the intermediate event is partly inside an ADS-B gap. |
| `ad963b` | ADS-B shows the aircraft landing at KAPF around 21:41; df1 says 22:43. |
| `adccd0` | The KPHL-KHPN leg departs around 17:43; df1 incorrectly reuses 14:03. |

### Both df0 and df1 are incorrect

| ICAO | Reason |
|---|---|
| `a5a744` | df1 combines an actual intermediate event into KORK-KORK, while ADSB.lol splits at the wrong intermediate airport/time (`KSUZ` rather than the ADSB-X-supported `KM89` event). |
| `a654ec` | df1 omits the earlier KFFZ local flight, while df0 misses the later KPRC touch-and-go and combines KFFZ-KPRC-KFFZ. |

### Genuine ADSB.lol algorithm/df0 failures

The failures fall into three recurring patterns. Every ICAO below was cross-checked against ADSB-X and SFDPS.

| Failure pattern | ICAOs | What happened |
|---|---|---|
| Missing entire or partial valid flight | `48d409`, `a00128`, `a003a8`, `a06abb`, `a099fd`, `a12110`, `a1c2f2`, `a287da`, `a35b42`, `a3c549`, `a442da`, `a48e3a`, `a50f17`, `a56d20`, `a593a0`, `a5ca4b`, `a689ee`, `a69b92`, `a6f149`, `a7d7b5`, `a7f27f`, `a81525`, `a871d4`, `aa9a0a`, `ac2bd5`, `ac2bd6`, `ad2170` | The valid SFDPS leg is independently present in ADSB-X but absent or incomplete in the ADSB.lol algorithm output. |
| Multiple real legs merged into one flight | `a02413`, `a0af5a`, `a105e8`, `a1b162`, `a23406`, `a25ec5`, `a2cff3`, `a2f9c0`, `a3242e`, `a37e7b`, `a3813a`, `a417af`, `a4ba76`, `a4ca06`, `a4d52b`, `a4d5a1`, `a6594f`, `a68754`, `a6fe45`, `a75b1f`, `a76ffa`, `a83bc6`, `a87f72`, `a8a337`, `a95169`, `aa2cf3`, `aa4ce2`, `aa9d1a`, `aaa590`, `aab838`, `ab4f5d`, `ab579f`, `ac86f1`, `ad5ae1`, `ad92cf`, `ada302` | ADSB.lol fails to recognize an intermediate landing or stop and emits one continuous origin-to-final-destination flight. |
| False fragmentation or incorrect endpoints | `a00e86`, `a19634`, `a1cf9c`, `a2e6f9`, `a34880`, `a597f2`, `a75d85`, `a98490`, `a9e666`, `abd5d8`, `ace718`, `adab24` | Coverage gaps or partial tracks cause invented intermediate airports, premature takeoffs/landings, or a flight that starts/ends at the wrong airport. |

> **Comparison caveat:** the evaluation uses `compare_airports=False`, so a row can be labeled `both` even when df0 and df1 disagree about one or both airports. The manual classifications above use the complete aircraft timeline rather than relying only on `match_status`.